## Root Cause Analysis - Exp4

* Goals
  * Check whether can provide improvement suggestions at individual cases level


In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import psycopg2
from utils_root_cause import upload_to_google_drive

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# Environment variable holding the database URL
DB_ENV = "DATABASE_URL_PUBLIC_RAG_DR"
db_url = os.getenv(DB_ENV)
if not db_url:
    raise RuntimeError(f"Environment variable {DB_ENV} is not set")

# Connect and load the table into a pandas DataFrame
conn = psycopg2.connect(db_url)
try:
    df_existing_rca_output = pd.read_sql_query("SELECT * FROM existing_rca_output", conn)
finally:
    try:
        conn.close()
    except Exception:
        pass

# Display the first rows
print(df_existing_rca_output.shape)
df_existing_rca_output.head()

C:\Users\wuhan\AppData\Local\Temp\ipykernel_17640\1673439458.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_existing_rca_output = pd.read_sql_query("SELECT * FROM existing_rca_output", conn)


(4, 5)


,id,config_hash,rca_records,agg_review,created_at
0,1,93f67870e1be18f0328798b37faf07df,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586
1,2,fac62bbce5555de5ca44b01442baea3f,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Retrieval is high qua...,2026-04-13 04:28:22.134432
2,3,cda0e838de11a82a1802e38b8b0d9ab9,[{'query': 'How to deposit a cheque issued to ...,"{'root_cause_analysis': 'Low semantic weight, ...",2026-04-13 17:09:46.469203
3,4,9c99877c5c2cde6ac3a5c05ea242b577,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Answer quality suffer...,2026-04-13 18:51:48.106385


In [3]:
# Convert any list-valued columns so each item becomes its own row
df = df_existing_rca_output.copy()
# Find columns that contain lists in at least one row
list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, list)).any()]
if not list_cols:
    print('No list-valued columns found; no conversion needed')
else:
    for col in list_cols:
        # Explode the list so each element becomes a separate row
        df = df.explode(col).reset_index(drop=True)
        # If the exploded elements are dict-like, expand their keys into columns
        if df[col].dropna().apply(lambda x: isinstance(x, dict)).any():
            expanded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
            df = pd.concat([df.drop(columns=[col]).reset_index(drop=True), expanded.reset_index(drop=True)], axis=1)
    df_records_rows = df
    print('Converted list-valued columns to rows. Result shape:', df_records_rows.shape)
    display(df_records_rows.head())

Converted list-valued columns to rows. Result shape: (120, 19)


,id,config_hash,agg_review,created_at,rca_records.query,rca_records.context,rca_records.ai_answer,rca_records.aq_reasoning,rca_records.rq_reasoning,rca_records.same_context,rca_records.needs_re_eval,rca_records.query_quality,rca_records.referenced_answer,rca_records.retrieved_content,rca_records.root_cause_analysis,rca_records.answer_quality_score,rca_records.retrieval_quality_score,rca_records.new_answer_quality_score,rca_records.new_retrieval_quality_score
0,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Have the associate endorse the back of the che...,The AI’s answer is fully relevant to the user’...,The retrieved content is identical to the cont...,True,0,clear,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,[],3,3,3,3
1,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Yes. You can purchase a USPS money order and l...,The AI’s answer fully addresses the user’s que...,The retrieved content matches the context exac...,True,0,clear,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,[],3,3,3,3
2,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,You can operate several “doing‑business‑as” (D...,The AI’s answer directly addresses the user’s ...,The retrieved content is identical to the cont...,True,0,ambiguous,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,[{'Please review query quality': ['Can one EIN...,3,3,3,3
3,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,"To obtain business credit, follow these practi...",The AI’s answer is fully relevant to the user’...,The retrieved content reproduces the entire co...,True,0,clear,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,[],3,3,3,3
4,1,93f67870e1be18f0328798b37faf07df,{'root_cause_analysis': 'Limited retrieval (To...,2026-04-13 04:09:33.082586,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,"When a company shuts down, the retirement plan...",The AI’s answer correctly explains the rollove...,The retrieved content mirrors the context exac...,False,0,clear,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,[{'auto eval score issue': 'score vs reasoning...,1,3,2,3


In [5]:
folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')
upload_to_google_drive(df_records_rows, folder_id, 'rca_revisit_v1.xlsx')

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=799250539083-ihi7rtknhm6oq5niabbk1nh6bi506air.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A64941%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=JC1kpHgRZhK9GLimGo6Rg3XCHC4Gwd&code_challenge=-kz5GMlECWvnE1ulSETSm_gW1PYztMEzHXb7Asx-lRA&code_challenge_method=S256&access_type=offline&prompt=consent
Uploaded file ID: 1PQyr8-ocML3TKCChTisH-BqYh-H6VHQp
